# Lab 2 — Machine Learning para Séries Temporais (notebook)

Versão enxuta e executável do Lab 2, sem firulas gráficas. Fio condutor:
**produção de veículos no Brasil** (Banco Central, SGS 1373).

Organização:

- **Parte 1** — ML "tabular": feature engineering (lags, rolling, calendário, FFT) + modelos.
- **Parte 2** — modelos com a ordem na arquitetura: ARIMA, ETS, ARIMA-Boost.
- Comparação com validação cruzada temporal honesta + previsão final.

> No Colab: rode antes `!pip install statsforecast scikit-learn statsmodels`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.fft import rfft, rfftfreq
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive

## 1. Dados — SGS 1373 (produção de veículos, mensal)

In [ ]:
url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.1373/dados?formato=csv&dataInicial=01/01/2012"
df = pd.read_csv(url, sep=";", decimal=",")
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df = df.set_index("data").sort_index().rename(columns={"valor": "y"})
df = df.asfreq("MS")
df["y"] = df["y"].interpolate()

print(f"{len(df)} meses, de {df.index.min().date()} a {df.index.max().date()}")
df.tail(3)

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(df.index, df["y"])
plt.title("Producao de veiculos no Brasil - SGS 1373")
plt.ylabel("Unidades/mes")
plt.grid(alpha=0.3)
plt.show()

## 2. Feature engineering

Lags (curtos + sazonais), janelas móveis (sempre com `shift(1)` para não vazar
o presente), calendário cíclico e uma dummy para o choque da COVID
(fábricas paradas em 2020).

In [ ]:
def construir_features(s: pd.Series) -> pd.DataFrame:
    """Lags + rolling + calendario + dummy COVID. Tudo causal."""
    X = pd.DataFrame(index=s.index)

    for lag in [1, 2, 3, 6, 12, 24]:
        X[f"lag_{lag}"] = s.shift(lag)

    for win in [3, 6, 12]:
        X[f"roll_mean_{win}"] = s.shift(1).rolling(win).mean()
        X[f"roll_std_{win}"]  = s.shift(1).rolling(win).std()

    mes = s.index.month
    X["mes_sin"] = np.sin(2 * np.pi * mes / 12)
    X["mes_cos"] = np.cos(2 * np.pi * mes / 12)

    X["covid"] = ((s.index >= "2020-03-01") & (s.index <= "2020-06-01")).astype(int)
    return X

X_full = construir_features(df["y"])
X_full.head(13)

### FFT — periodograma

A Transformada de Fourier revela quais ciclos dominam a série. Tiramos a
tendência antes para a FFT enxergar melhor o ciclo.

In [ ]:
y = df["y"].values.astype(float)
y_detrend = y - np.polyval(np.polyfit(np.arange(len(y)), y, 1), np.arange(len(y)))

amp = np.abs(rfft(y_detrend - y_detrend.mean()))
freq = rfftfreq(len(y_detrend), d=1)
periodo = np.divide(1, freq, where=freq > 0)

mask = (periodo >= 2) & (periodo <= 48)
top = periodo[mask][np.argmax(amp[mask])]

plt.figure(figsize=(10, 4))
plt.plot(periodo[mask], amp[mask])
plt.axvline(top, color="green", ls="--", label=f"Pico ~{top:.1f} meses")
plt.xlabel("Periodo (meses por ciclo)")
plt.ylabel("Amplitude")
plt.title("Periodograma")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Ciclo dominante: ~{top:.1f} meses (esperado: 12)")

## 3. Parte 1 — modelo de ML "tabular" (GradientBoosting)

Validação cruzada *expanding window*: 4 janelas de 12 meses.

In [ ]:
def cv_boost(s, n_windows=4, h=12):
    X = construir_features(s).dropna()
    y = s.loc[X.index]
    n = len(X)
    test_starts = [n - (n_windows - i) * h for i in range(n_windows)]
    rows = []
    for i, start in enumerate(test_starts):
        m = GradientBoostingRegressor(
            n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42
        ).fit(X.iloc[:start], y.iloc[:start])
        rows.append(pd.DataFrame({
            "ds": X.index[start:start+h], "y": y.iloc[start:start+h].values,
            "boost": m.predict(X.iloc[start:start+h]), "window": i,
        }))
    return pd.concat(rows, ignore_index=True)

cv_b = cv_boost(df["y"])
cv_b.head(3)

In [ ]:
X_fit = construir_features(df["y"]).dropna()
y_fit = df["y"].loc[X_fit.index]
modelo = GradientBoostingRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42
).fit(X_fit, y_fit)

imp = pd.Series(modelo.feature_importances_, index=X_fit.columns).sort_values().tail(10)
imp.plot.barh(figsize=(8, 4), title="Top features (GradientBoosting)")
plt.tight_layout()
plt.show()

### Frameworks modernos (referência — não executa aqui)

Em produção você não escreve a CV à mão. Os dois frameworks dominantes:

```python
# MLForecast (Nixtla) — pip install mlforecast lightgbm
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from lightgbm import LGBMRegressor

sf_df = (df.reset_index().rename(columns={"data": "ds"})
           .assign(unique_id="veiculos")[["unique_id", "ds", "y"]])
mlf = MLForecast(
    models={"lgbm": LGBMRegressor(n_estimators=300, learning_rate=0.05)},
    freq="MS", lags=[1, 2, 3, 12, 24],
    lag_transforms={1: [RollingMean(3), RollingMean(12)]},
    date_features=["month", "quarter"],
)
cv_mlf = mlf.cross_validation(df=sf_df, h=12, n_windows=4)

# DARTS — pip install u8darts[all]
from darts import TimeSeries
from darts.models import LightGBMModel
serie = TimeSeries.from_dataframe(df.reset_index(), "data", "y", freq="MS")
m = LightGBMModel(lags=[-1, -2, -3, -12, -24], output_chunk_length=12)
m.fit(serie[:-24]); prev = m.predict(n=24)
```

## 4. Parte 2 — modelos com a ordem na arquitetura

ARIMA, ETS e SeasonalNaive via `statsforecast` (AutoARIMA/AutoETS escolhem
ordens/forma por AICc).

In [ ]:
sf_df = (df.reset_index().rename(columns={"data": "ds"})
           .assign(unique_id="veiculos")[["unique_id", "ds", "y"]])

sf = StatsForecast(
    models=[
        AutoARIMA(season_length=12, alias="arima"),
        AutoETS(season_length=12, alias="ets"),
        SeasonalNaive(season_length=12, alias="snaive"),
    ],
    freq="MS", n_jobs=1,
)
cv_sf = sf.cross_validation(df=sf_df, h=12, step_size=12, n_windows=4).reset_index()
cv_sf.head(3)

### ARIMA-Boost (híbrido)

SARIMA captura tendência+sazonalidade; o boosting aprende os resíduos
não-lineares.

In [ ]:
def cv_arima_boost(s, n_windows=4, h=12, order=(1,1,1), seas=(0,1,1,12)):
    X_all = construir_features(s)
    n = len(s)
    test_starts = [n - (n_windows - i) * h for i in range(n_windows)]
    rows = []
    for i, start in enumerate(test_starts):
        s_tr = s.iloc[:start]
        sarima = SARIMAX(s_tr, order=order, seasonal_order=seas).fit(disp=False)
        prev_arima = sarima.forecast(steps=h)

        resid = (s_tr - sarima.fittedvalues).rename("r")
        X_tr = X_all.loc[resid.index].dropna()
        r_tr = resid.loc[X_tr.index]

        boost = GradientBoostingRegressor(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        ).fit(X_tr, r_tr)

        X_te = X_all.iloc[start:start+h]
        rows.append(pd.DataFrame({
            "ds": X_te.index, "y": s.iloc[start:start+h].values,
            "arima_boost": prev_arima.values + boost.predict(X_te), "window": i,
        }))
    return pd.concat(rows, ignore_index=True)

cv_h = cv_arima_boost(df["y"])
cv_h.head(3)

## 5. Comparação com validação honesta

MASE < 1 = ganhei do baseline naive sazonal; > 1 = perdi.

In [ ]:
def mase(y_true, y_pred, y_train, m=12):
    naive = np.mean(np.abs(y_train[m:] - y_train[:-m]))
    return mean_absolute_error(y_true, y_pred) / naive

def metricas(y_true, y_pred, y_train):
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MASE": mase(y_true, y_pred, y_train),
    }

eval_df = (
    cv_sf[["ds", "y", "arima", "ets", "snaive"]]
    .merge(cv_b[["ds", "boost"]], on="ds")
    .merge(cv_h[["ds", "arima_boost"]], on="ds")
)
y_train = df["y"].iloc[:-48].values
resumo = pd.DataFrame({
    nome: metricas(eval_df["y"], eval_df[nome], y_train)
    for nome in ["snaive", "ets", "arima", "boost", "arima_boost"]
}).T.sort_values("MASE")
resumo.round(1)

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(df.index[-60:], df["y"].iloc[-60:], color="gray", label="Observado")
for nome in ["arima", "boost", "arima_boost"]:
    plt.plot(eval_df["ds"], eval_df[nome], label=nome, alpha=0.85)
plt.title("Cross-validation em 4 janelas")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Previsão final dos próximos 12 meses

Reajusta o melhor candidato estatístico (ARIMA/ETS) em toda a série e
projeta 12 meses com intervalo de 95%.

In [ ]:
sf_final = StatsForecast(
    models=[AutoARIMA(season_length=12, alias="arima"),
            AutoETS(season_length=12, alias="ets")],
    freq="MS", n_jobs=1,
)
fc = sf_final.forecast(df=sf_df, h=12, level=[95])
fc.tail(12)

In [ ]:
plt.figure(figsize=(11, 4.5))
plt.plot(df.index[-36:], df["y"].iloc[-36:], color="gray", label="Observado")
plt.plot(fc["ds"], fc["arima"], label="Previsao ARIMA")
plt.fill_between(fc["ds"], fc["arima-lo-95"], fc["arima-hi-95"], alpha=0.2)
plt.title("Previsao 12 meses - producao de veiculos")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Foundation models (referência — não executa aqui)

Modelos pré-treinados que fazem *zero-shot forecasting* (sem treinar):

```python
# TimeGPT (Nixtla) — pip install nixtla ; requer API key
from nixtla import NixtlaClient
client = NixtlaClient(api_key="SUA_CHAVE")
forecast = client.forecast(df=sf_df, h=12, freq="MS")

# Chronos (Amazon) — pip install chronos-forecasting ; open-source local
import torch
from chronos import ChronosPipeline
pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-small")
fc = pipe.predict(torch.tensor(df["y"].values), prediction_length=12)
```

Vantagens: baseline forte sem treino, ótimo para prototipagem e muitas séries.
Desvantagens: caixa-preta, fraco em domínios atípicos, custo/infra.

Material completo, diagramas e referências: ver a página do **Lab 2**.